# RailGuard AI — YOLO11s V2 GPU Training & Evaluation (Google Colab)

This notebook provides a complete GPU-accelerated pipeline for training and evaluating **YOLO11s V2** for the RailGuard AI system.

--- 
### 📌 Pipeline Overview
1. **Environment Setup**: GPU verification & package installations (`ultralytics`, `torch`, `opencv`, etc.)
2. **Dataset Setup**: Upload/Extract railway defect dataset & configure `data.yaml`
3. **YOLO11s V2 Training**: 50-Epoch AdamW training with full augmentation (`imgsz=640`, `device=0`)
4. **Validation & Test Evaluation**: Compute Precision, Recall, $mAP_{50}$, $mAP_{50-95}$, Confusion Matrix, PR/F1 Curves & all 15 Per-Class metrics
5. **V1 vs V2 Comparison**: Benchmark V2 against V1 baseline metrics
6. **Deployment Safeguard**: Deployment gate remains locked — download `best.pt` for local verification.

## 1. Environment & GPU Verification

In [ ]:
# Install Ultralytics and dependencies
!pip install -q ultralytics torch torchvision opencv-python pandas matplotlib seaborn pyyaml

import torch
import ultralytics
from ultralytics import YOLO
import sys, os, time, json
import pandas as pd
import matplotlib.pyplot as plt

print(f"Python Version    : {sys.version.split()[0]}")
print(f"PyTorch Version   : {torch.__version__}")
print(f"Ultralytics       : {ultralytics.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device Name   : {torch.cuda.get_device_name(0)}")
    print(f"GPU Count         : {torch.cuda.device_count()}")
else:
    print("⚠️ WARNING: CUDA GPU not detected! Please go to Runtime -> Change runtime type -> Select T4 GPU.")

## 2. Dataset Setup & YAML Configuration

Upload `railway-defect-dataset.zip` directly to Colab or mount Google Drive.

In [ ]:
from pathlib import Path
import zipfile

# Define dataset directory in Colab workspace
DATASET_DIR = Path("/content/railway-defect-dataset")
ZIP_PATH = Path("/content/railway-defect-dataset.zip")

# Ensure dataset directory exists
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Option A: If uploaded zip file exists, extract directly into DATASET_DIR
if ZIP_PATH.exists():
    print(f"Extracting {ZIP_PATH} into {DATASET_DIR} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)
    print("✅ Extraction complete.")
elif (Path("/content/drive/MyDrive/railway-defect-dataset.zip")).exists():
    drive_zip = Path("/content/drive/MyDrive/railway-defect-dataset.zip")
    print(f"Extracting from Google Drive: {drive_zip} ...")
    with zipfile.ZipFile(drive_zip, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)
    print("✅ Extraction complete from Drive.")
elif not (DATASET_DIR / "train" / "images").exists():
    print("ℹ️ Dataset zip not found at /content/railway-defect-dataset.zip")
    print("Please upload railway-defect-dataset.zip to /content/ via the left file browser, or mount Drive:")
    print("from google.colab import drive; drive.mount('/content/drive')")

# Create data.yaml configured for Colab paths
DATA_YAML_CONTENT = f"""path: {DATASET_DIR.as_posix()}
train: train/images
val: valid/images
test: test/images

names:
  0: broken_rail_high
  1: broken_rail_low
  2: broken_rail_medium
  3: corrosion_high
  4: corrosion_low
  5: corrosion_medium
  6: head_checks_high
  7: head_checks_low
  8: head_checks_medium
  9: railway_join_bar_defects_high
  10: railway_join_bar_defects_low
  11: railway_join_bar_defects_medium
  12: squats_high
  13: squats_low
  14: squats_medium
"""

yaml_file = Path("/content/data.yaml")
yaml_file.write_text(DATA_YAML_CONTENT)
print(f"\n✅ Created dynamic data.yaml at: {yaml_file}")

## 3. YOLO11s V2 Training Execution (GPU Accelerated)

Executes full 50-Epoch training with early stopping (`patience=20`) and optimized augmentations.

In [ ]:
# Initialize YOLO11s pretrained weights
model = YOLO('yolo11s.pt')

RUNS_DIR = Path('/content/runs')

t_start = time.time()
results = model.train(
    data=str(yaml_file),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,                   # GPU 0
    workers=4,
    project=str(RUNS_DIR),
    name='yolo11s_railguard_v2',
    save=True,
    save_period=5,
    exist_ok=True,
    
    # Optimizer
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    cos_lr=True,
    
    # Early Stopping
    patience=20,
    
    # Loss Weights
    cls=1.0,
    box=7.5,
    dfl=1.5,
    
    # Augmentations
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    
    val=True,
    plots=True,
    verbose=True
)

total_time = time.time() - t_start
print(f"\n✅ GPU Training Completed in {total_time/60:.2f} minutes!")

## 4. Model Evaluation & Per-Class Metric Extraction

In [ ]:
BEST_PT = RUNS_DIR / 'yolo11s_railguard_v2' / 'weights' / 'best.pt'
assert BEST_PT.exists(), f"Missing trained model weights at {BEST_PT}"

best_model = YOLO(str(BEST_PT))

# Evaluate on Validation & Test Splits
print("Evaluating on VALIDATION split...")
val_res = best_model.val(data=str(yaml_file), split='val', imgsz=640, device=0, plots=True)

print("\nEvaluating on TEST split...")
test_res = best_model.val(data=str(yaml_file), split='test', imgsz=640, device=0, plots=True)

print("\n" + "="*65)
print("  YOLO11s V2 Test Metrics Summary")
print("="*65)
rd = test_res.results_dict
v2_p   = rd.get('metrics/precision(B)', 0)
v2_r   = rd.get('metrics/recall(B)', 0)
v2_m50 = rd.get('metrics/mAP50(B)', 0)
v2_m95 = rd.get('metrics/mAP50-95(B)', 0)

print(f"Precision  : {v2_p:.4f}  ({v2_p*100:.2f}%)")
print(f"Recall     : {v2_r:.4f}  ({v2_r*100:.2f}%)")
print(f"mAP50      : {v2_m50:.4f}  ({v2_m50*100:.2f}%)")
print(f"mAP50-95   : {v2_m95:.4f}  ({v2_m95*100:.2f}%)")

# ── Per-Class Metrics Table (All 15 Defect Classes) ─────────────
print("\n" + "-"*65)
print(f"  {'Class Name':<32} {'P':>7} {'R':>7} {'mAP50':>8} {'mAP50-95':>10}")
print("-"*65)
if hasattr(test_res, 'ap_class_index') and test_res.ap_class_index is not None:
    names = best_model.names
    for i, cls_id in enumerate(test_res.ap_class_index):
        cname = names.get(int(cls_id), f'cls_{cls_id}')
        p = test_res.box.p[i] if i < len(test_res.box.p) else 0.0
        r = test_res.box.r[i] if i < len(test_res.box.r) else 0.0
        ap50 = test_res.box.ap50[i] if i < len(test_res.box.ap50) else 0.0
        ap5095 = test_res.box.ap[i] if i < len(test_res.box.ap) else 0.0
        print(f"  {cname:<32} {p:>7.4f} {r:>7.4f} {ap50:>8.4f} {ap5095:>10.4f}")
print("-"*65)

## 5. Benchmark Comparison: V1 vs V2

In [ ]:
# V1 Baseline Metrics (Recorded)
v1_metrics = {
    'Model': 'YOLO11n V1 (Baseline)',
    'Precision': 0.4652,
    'Recall': 0.2347,
    'mAP50': 0.1881,
    'mAP50-95': 0.0728
}

v2_metrics = {
    'Model': 'YOLO11s V2 (GPU Trained)',
    'Precision': v2_p,
    'Recall': v2_r,
    'mAP50': v2_m50,
    'mAP50-95': v2_m95
}

comparison_df = pd.DataFrame([v1_metrics, v2_metrics])
print("\n=== RailGuard AI Model Performance Comparison ===")
print(comparison_df.to_string(index=False))

## 6. Deployment Gate Safeguard & Artifact Export

🔒 **Deployment Gate Locked**: Do NOT copy `best.pt` directly to `ai-service/models/` until local evaluation is complete.
Download `best.pt` and `results.csv` to transfer to your local RailGuard AI repository.

In [ ]:
import shutil
from google.colab import files

# 1. Package all run artifacts into a single portable zip (weights, metrics, plots)
artifacts_dir = RUNS_DIR / 'yolo11s_railguard_v2'
archive_zip = Path('/content/yolo11s_railguard_v2_artifacts')

print(f"Compressing training artifacts from {artifacts_dir} ...")
shutil.make_archive(str(archive_zip), 'zip', str(artifacts_dir))
print(f"✅ Archive created: {archive_zip}.zip")

# 2. Trigger download of the complete artifacts bundle
print("Triggering download of artifacts bundle (includes best.pt, results.csv, plots) ...")
files.download(f"{archive_zip}.zip")

# 3. Also download best.pt directly for convenience
print(f"best.pt is located at: {BEST_PT}")
files.download(str(BEST_PT))